In [1]:
prompt = "Indeed Ankara was proclaimed the capital in "

next_token ="1923"

output = "Indeed Ankara was proclaimed the capital in 1923"

In [2]:
data = "Indeed Ankara was proclaimed the capital"
hedef ="was proclaimed the capital in 1923"

In [3]:
from tokenizer import Tokenizer

tokenizer = Tokenizer("tokenizer.json")
ids = tokenizer.encode(hedef)

len(ids),ids

(12, [52, 2, 120, 105, 2, 14, 2, 48, 2, 33, 2, 127])

In [4]:
context_length = 12 #gpt-4o 128k, gemma-3-1b 32k

In [5]:

ids = tokenizer.encode(prompt)

len(ids),ids

(14, [126, 2, 77, 2, 52, 2, 120, 105, 2, 14, 2, 48, 2, 33])

In [6]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM
import os

hf_token = os.getenv("HF_TOKEN")
model_id = "google/gemma-3-1b-it"

gemma_tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
gemma_model = AutoModelForCausalLM.from_pretrained(model_id, token=hf_token)
print("ok")

c:\Models\LLM-From-Scratch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 340/340 [00:00<00:00, 4615.66it/s]


ok


In [7]:
# !pip install transformers huggingface_hub sentencepiece accelerate -q

In [8]:
# import sys
# print(sys.executable)

In [9]:

# import sys
# !{sys.executable} -m pip install transformers huggingface_hub sentencepiece accelerate

In [10]:
gemma_model

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1152, out_features=256, bias=False)
          (v_proj): Linear(in_features=1152, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((1152,), e

In [11]:
girdi = "Indeed Ankara was proclaimed the capital"
cikti = "was proclaimed the capital in 1923"

In [12]:
with open("text.txt","r") as f:
    text = f.read()

text

'This work is addressing the Capital issue as a symbol of nation-state building, which contains the founding philosophy of the Republic of Turkey after the victory in the war of independence (Milli MÃ¼cadele). Especially Mustafa Kemal did not want to continue with Istanbul as the capital of the new Republic, since it was representing the Ottoman Empire. Thus, the fundamental reasons underlying this choice was the desire to establish a new state in creation of whose identity a capital would play a defining role, as well as the troublesome situation that the Istanbul was in. Even though Ankara had already begun bearing the responsibility of a capital ever since the Representative Committee (Heyet-i Temsiliye) was moved from Sivas to Ankara, it was not expecting to be a capital, considering it a temporary situation. On the other hand, Gazi Mustafa Kemal had given the impression at the very outset that he wanted to see Ankara as the new capital. Particularly after the liberation of Istanbu

In [13]:
token_ids = tokenizer.encode(text)
#save ids
ids_text =""

for token_id in token_ids:
    ids_text += f"{token_id} "

with open("token_ids.txt","w") as f:
    f.write(ids_text)

In [14]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cpu
False


In [15]:
from text_dataset import create_data_loader

stride = 5
train_data_loader = create_data_loader(token_ids, context_length, batch_size=4, stride=stride, shuffle=True, device="cpu")
len(train_data_loader)

37

In [16]:
i = 0
for batch in enumerate(train_data_loader):
    print(batch)
    i += 1
    if i >=2:
        break
#batch_size kadar input ve target

(0, [tensor([[ 14,   2, 137,   2, 138,   4,   2, 139,   4,   2, 140,   2],
        [  2,  48,   2,  83,   2,  50,   2,  14,   2,  84,   2,  85],
        [ 60,   2,  61,   2,  52,   2,  14,   2,  62,   2,  44,   2],
        [148,   4,   2,  39,   2,  40,   2,  52,   2, 149,   1,   2]]), tensor([[  2, 137,   2, 138,   4,   2, 139,   4,   2, 140,   2, 115],
        [ 48,   2,  83,   2,  50,   2,  14,   2,  84,   2,  85,   2],
        [  2,  61,   2,  52,   2,  14,   2,  62,   2,  44,   2,  63],
        [  4,   2,  39,   2,  40,   2,  52,   2, 149,   1,   2,  33]])])
(1, [tensor([[  2, 114,   2,  52,   2, 145,  13,   2,  74,   2,  60,   2],
        [ 89,   2,  90,   2,  44,   2,  77,   4,   2,  51,   2,  52],
        [ 14,   2,  49,   2,  48,   3,   2, 107,   2,  31,   2,  14],
        [  2,  18,   2, 167,   2, 168,   2,  20,   2,  60,   2,  10]]), tensor([[114,   2,  52,   2, 145,  13,   2,  74,   2,  60,   2,  67],
        [  2,  90,   2,  44,   2,  77,   4,   2,  51,   2,  52,   2],
   

In [17]:
data_iter = iter(train_data_loader)

In [24]:
next(data_iter)

[tensor([[26,  2, 59, 13,  2, 60,  2, 61,  2, 52,  2, 14],
         [14,  2, 48,  4,  2, 39,  2, 40,  2, 52,  2,  1],
         [14,  2, 29,  2, 20,  2, 30,  2, 31,  2, 14,  2],
         [74,  2, 77,  2, 67,  2, 92,  2, 14,  2, 49,  2]]),
 tensor([[ 2, 59, 13,  2, 60,  2, 61,  2, 52,  2, 14,  2],
         [ 2, 48,  4,  2, 39,  2, 40,  2, 52,  2,  1,  1],
         [ 2, 29,  2, 20,  2, 30,  2, 31,  2, 14,  2, 32],
         [ 2, 77,  2, 67,  2, 92,  2, 14,  2, 49,  2,  1]])]

Embedding yani sözlük anlamlarının sayısal değerleri

In [84]:
import plotly.graph_objects as go
import plotly.offline

def plot_dots(sentences_data, title):
    data = [
        go.Scatter3d(
            x=sentence_data["words"][:, 0],
            y=sentence_data["words"][:, 1],
            z=sentence_data["words"][:, 2],
            mode="markers+text",
            marker=dict(
                size=10,
                color=sentence_data["color"]
            ),
            text=sentence_data["labels"],
            hoverinfo="text"
        )
        for sentence_data in sentences_data
    ]

    layout = go.Layout(
        scene=dict(
            xaxis_title="Sertlik",
            yaxis_title="Parlaklık",
            zaxis_title="Kırmızılık",
        ),
        title=title,
    )

    fig = go.Figure(data=data, layout=layout)
    plotly.offline.plot(fig)

In [85]:
import numpy as np
sentences = [
    {
        "words": np.array([
            [0.1, 0.20, 0.23, 0.0],
            [0.4, 0.2, 0.4, 0.3],
            [0.12, 0.24, 0.2, 0.1],
            [0.0, 0.0, 0.3, 0.01],
            [0.0, 0.0, 0.0, 0.0]
        ]),
        "labels": ["Indeed", "Ankara", "was", "proclaimed"],
        "color": "red",
    },
    {
        "words": np.array([
            [0.1, 0.2, 0.4, 0.1],
            [0.12, 0.24, 0.2, 0.1],
            [0.0, 0.0, 0.3, 0.1],
            [0.0, 0.0, 0.0, 0.0]
        ]),
        "labels": ["Ankara", "was", "proclaimed", "the"],
        "color": "blue",
    }
]

plot_dots(sentences, "Sözlük V1")

In [86]:
vocab_size = max(tokens) + 1
embeddings = torch.nn.Embedding(vocab_size, 4)

In [87]:
tokens = tokenizer.encode("Indeed Ankara was proclaimed the")
tokens

[126, 2, 77, 2, 52, 2, 120, 105, 2, 14]

In [88]:
meanings = embeddings(torch.tensor(tokens))
meanings.shape

torch.Size([10, 4])

In [89]:
sentences = [
    {
        "words": meanings.detach().numpy(),
        "labels": ["Indeed", "Ankara", "was", "proclaimed", "the"],
        "color": "red",
    },
]

plot_dots(sentences, "Sözlük V1")

In [90]:
gemma_tokens = gemma_tokenizer.encode("Indeed Ankara was proclaimed the")
gemma_tokens

[2, 51077, 80554, 691, 72970, 506]

In [91]:
gemma_meanings = gemma_model.get_input_embeddings()(torch.tensor(gemma_tokens))
gemma_meanings.shape

torch.Size([6, 1152])

In [92]:
gemma_sentences = [
    {
        "words": gemma_meanings.detach().numpy(),
        "labels": gemma_tokenizer.tokenize("the capital of united states"),
        "color": "red",
    },
]

plot_dots(gemma_sentences, "Gemma", dims=[20, 21, 22])

TypeError: Got unsupported ScalarType BFloat16